# 🕸️ 01-数据结构与算法 · 主题 10：图（BFS/DFS/拓扑/并查集/最短路径/Trie）

> 图论题在秋招里占比逐年升高：网格 DFS、拓扑排序、并查集、最短路是四大高频模板。
> 本 notebook 目标：**掌握两种表示 → 背熟四大模板 → 吃透 8 道高频题 → 通过 25 道自测**。

**前置**：主题 4（队列）、主题 6（树/DFS）。**运行环境**：Python 3.8+（用 `heapq`/`collections` 标准库）。

## §1 图的表示与复杂度

### 两种主流表示

| 表示 | 结构 | 查询边 | 空间 | 适用 |
|------|------|--------|------|------|
| 邻接矩阵 | `matrix[u][v]` | $O(1)$ | $O(V^2)$ | 稠密图、小规模 |
| 邻接表 | `adj[u] = [v,...]` | $O(deg(u))$ | $O(V+E)$ | **绝大多数题** |

Python 邻接表常用写法：

```python
adj = [[] for _ in range(n)]
for u, v in edges:
    adj[u].append(v)
    adj[v].append(u)      # 无向图加双向
```

### 遍历复杂度

$$\text{DFS/BFS 时间} = O(V + E)$$

每个点进队列/栈一次、每条边被访问一次。

### 有向 vs 无向 / 有权 vs 无权

- **无权图最短路径** → BFS（队列）
- **有权图最短路径** → Dijkstra（堆）
- **有向无环（DAG）** → 拓扑排序
- **连通性/环** → 并查集 / DFS

## §2 BFS 与 DFS 模板（背熟）

### BFS（逐层扩展，最短路径）

```python
from collections import deque
q = deque([起点]); visited = {起点}
while q:
    node = q.popleft()
    for nb in adj[node]:
        if nb not in visited:
            visited.add(nb)
            q.append(nb)
```

**为什么 BFS 找最短**：第 $k$ 层出队的节点都是距离起点 $k$ 步的——首次访问即最短。

### DFS（深入探索，遍历/连通块）

```python
def dfs(u):
    visited.add(u)
    for nb in adj[u]:
        if nb not in visited:
            dfs(nb)
```

> ⚠️ **visited 去重是灵魂**：无向图必须去重，否则死循环；有向图也要防重复入队。

### 网格 BFS 模板（200 岛屿数量 / 1091 最短路径）

- 方向数组：`dirs = [(1, 0), (-1, 0), (0, 1), (0, -1)]`（下 / 上 / 右 / 左）
- 队列存三元组 `(r, c, d)`：`d` 为已走步数，**首次出队即最短路径**（1091 模板）
- **入队即标记 visited**：先 `visited.add` 再入队，防止同一格被重复入队（200 岛屿的"淹没"同理）
- 边界检查：`0 <= nr < R and 0 <= nc < C`，再判断 `grid[nr][nc]` 是否可走

In [ ]:
from collections import deque

def build_adj(n, edges, directed=False):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        if not directed:
            adj[v].append(u)
    return adj

# 例图：0-1, 0-2, 1-3, 2-3
adj = build_adj(4, [(0, 1), (0, 2), (1, 3), (2, 3)])

def bfs(start, adj):
    res = []
    q = deque([start])
    visited = {start}
    while q:
        u = q.popleft()
        res.append(u)
        for v in adj[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
    return res

def dfs(start, adj):
    res = []
    visited = set()
    def go(u):
        visited.add(u)
        res.append(u)
        for v in adj[u]:
            if v not in visited:
                go(v)
    go(start)
    return res

print('BFS from 0:', bfs(0, adj))    # 期望 [0, 1, 2, 3]
print('DFS from 0:', dfs(0, adj))    # 期望 [0, 1, 3, 2]（按邻接表顺序）

## §3 岛屿数量（LeetCode 200）：连通块计数

**题目**：`1` 表示陆地，求连通的陆地块数（上下左右）。

**思路**：遍历网格，遇到未访问的 `1` 就 DFS/BFS 淹没整块并计数：

$$\text{岛屿数} = \sum_{\text{起点} \in \text{未访问陆地}} 1$$

**网格 DFS 四方向**：

$$(i \pm 1, j), \quad (i, j \pm 1)$$

**复杂度**：$O(R \times C)$（每个格子至多访问一次），空间 $O(R \times C)$ 最坏。

In [ ]:
def num_islands(grid):
    R, C = len(grid), len(grid[0])
    def sink(i, j):
        # 原地标记：把连通块全部变成 0，避免额外 visited
        if not (0 <= i < R and 0 <= j < C) or grid[i][j] == '0':
            return
        grid[i][j] = '0'
        sink(i + 1, j); sink(i - 1, j); sink(i, j + 1); sink(i, j - 1)

    count = 0
    for i in range(R):
        for j in range(C):
            if grid[i][j] == '1':
                count += 1
                sink(i, j)
    return count

g = [
    ['1', '1', '0', '0', '0'],
    ['1', '1', '0', '0', '0'],
    ['0', '0', '1', '0', '0'],
    ['0', '0', '0', '1', '1'],
]
print('num_islands =', num_islands(g))    # 期望 3
print('num_islands(全0) =', num_islands([['0', '0'], ['0', '0']]))   # 0
print('num_islands(全1) =', num_islands([['1', '1'], ['1', '1']]))   # 1

## §4 腐烂的橘子（994）：多源 BFS 分层

**题目**：烂橘子每分钟感染相邻好橘子，求全部烂掉所需分钟（不可能则 -1）。

**思路**：把**所有初始烂橘子**一起放进队列（多源 BFS），逐层扩散：

- 层数 = 分钟数（在出队时记录 `step`）
- 结束时若还有好橘子 → -1

**为什么多源 BFS**：多个源头同时扩散，等价于超级源点连接所有烂橘子 → 经典 BFS 单源模型。

**复杂度**：$O(R \times C)$。

In [ ]:
def oranges_rotting(grid):
    R, C = len(grid), len(grid[0])
    q = deque()
    fresh = 0
    for i in range(R):
        for j in range(C):
            if grid[i][j] == 2:
                q.append((i, j, 0))
            elif grid[i][j] == 1:
                fresh += 1
    minutes = 0
    while q:
        i, j, t = q.popleft()
        minutes = max(minutes, t)
        for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ni, nj = i + di, j + dj
            if 0 <= ni < R and 0 <= nj < C and grid[ni][nj] == 1:
                grid[ni][nj] = 2      # 感染
                fresh -= 1
                q.append((ni, nj, t + 1))
    return minutes if fresh == 0 else -1

print('test1:', oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]))   # 4
print('test2:', oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]))   # -1（角上孤立）
print('test3:', oranges_rotting([[0, 2]]))                            # 0（无新鲜）
print('test4:', oranges_rotting([[2, 2, 2], [2, 2, 2]]))              # 0

## §5 拓扑排序（207 课程表）：Kahn 算法

**问题**：判断有向图是否有环（能否完成所有课程）。

**Kahn 算法**（BFS 版）：

1. 统计每个节点的入度
2. 入度为 0 的节点入队（无前置依赖）
3. 出队时把它的所有后继入度 -1，减到 0 就入队
4. **出队总数 < 节点数 → 有环**（环上节点永远入度 > 0）

**复杂度**：$O(V + E)$。

> 💡 拓扑序不唯一；「字典序最小」拓扑序 = 用最小堆替换队列。

In [ ]:
def can_finish(num_courses, prerequisites):
    adj = [[] for _ in range(num_courses)]
    indeg = [0] * num_courses
    for a, b in prerequisites:          # b 是 a 的前置
        adj[b].append(a)
        indeg[a] += 1
    q = deque([i for i in range(num_courses) if indeg[i] == 0])
    done = 0
    while q:
        u = q.popleft()
        done += 1
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return done == num_courses          # 全部出队 = 无环

print('test1:', can_finish(2, [[1, 0]]))                 # True
print('test2:', can_finish(2, [[1, 0], [0, 1]]))         # False（环）
print('test3:', can_finish(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))  # True
print('test4:', can_finish(1, []))                       # True

## §5.1 课程表 II（210）：输出拓扑序列

**题目**：给定课程数与先修关系，返回一种可行修课顺序；有环则返回 `[]`。

**与 207 的区别**：Kahn 算法把**出队节点收集进结果列表**；结果长度 == numCourses 则有解，否则有环返回空数组。

In [ ]:
from collections import deque

def find_order(num_courses, prerequisites):
    adj = [[] for _ in range(num_courses)]
    indeg = [0] * num_courses
    for a, b in prerequisites:              # b 是 a 的前置
        adj[b].append(a)
        indeg[a] += 1
    q = deque([i for i in range(num_courses) if indeg[i] == 0])
    order = []
    while q:
        u = q.popleft()
        order.append(u)                     # 出队即确定一个位置
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order if len(order) == num_courses else []   # 有环 → []

print('test1:', find_order(2, [[1, 0]]))                # [0, 1]
print('test2:', find_order(2, [[1, 0], [0, 1]]))        # []（环）
print('test3:', find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))  # [0, 1, 2, 3]

assert find_order(2, [[1, 0]]) == [0, 1]
assert find_order(2, [[1, 0], [0, 1]]) == []

## §6 并查集（Union-Find）：模板与 684 冗余连接

**用途**：动态合并集合、判断连通性，近似 $O(\alpha(n))$（反阿克曼函数，可视为常数）。

**两个优化**：

- **路径压缩**：find 时把沿途节点直接指向根
- **按秩合并**：矮树挂到高树上

### 684 冗余连接

**题目**：无向图中多加了一条边形成环，删掉它。

**思路**：依次并边——若 `find(u) == find(v)` 说明已连通，这条边就是冗余边。

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        # 路径压缩
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False          # 已连通（成环）
        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1
        return True

def find_redundant_connection(edges):
    uf = UnionFind(len(edges) + 1)
    for u, v in edges:
        if not uf.union(u, v):
            return [u, v]         # 第一条成环边
    return []

print('test1:', find_redundant_connection([[1, 2], [1, 3], [2, 3]]))            # [2,3]
print('test2:', find_redundant_connection([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]))  # [1,4]

# 547 省份数量：连通分量数
def find_circle_num(is_connected):
    n = len(is_connected)
    uf = UnionFind(n)
    for i in range(n):
        for j in range(i + 1, n):
            if is_connected[i][j]:
                uf.union(i, j)
    return len({uf.find(i) for i in range(n)})

print('省份数量:', find_circle_num([[1, 1, 0], [1, 1, 0], [0, 0, 1]]))   # 2
print('省份数量:', find_circle_num([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))   # 3

## §6.1 Kruskal 最小生成树

**思路**：边按**权值升序**排序，用并查集依次尝试连边——`find(u) != find(v)` 才合并并累加权值，已连通（成环）则跳过。连满 `n-1` 条边即成 MST。

**复杂度**：$O(E \log E)$（排序为主，并查集近常数）。

In [ ]:
def kruskal(n, edges):
    uf = UnionFind(n)                       # 复用 §6 的并查集（路径压缩 + 按秩合并）
    total = 0
    mst = []
    for u, v, w in sorted(edges, key=lambda e: e[2]):   # 按边权升序贪心
        if uf.union(u, v):                  # 不成环才合并
            total += w
            mst.append((u, v, w))
    return total, mst

n = 4
edges = [(0, 1, 1), (0, 2, 2), (1, 2, 3), (1, 3, 4), (2, 3, 5)]
total, mst = kruskal(n, edges)
print('总权:', total)                        # 7 = 1 + 2 + 4
print('MST 边集:', mst)

assert total == 7
assert len(mst) == n - 1

## §7 Dijkstra 最短路径（743 网络延迟时间）

**适用**：非负权单源最短路。

**思路**（贪心 + 堆）：每次从堆里取「当前距离最小」的未确定节点，松弛它的所有出边：

$$dist[v] = \min(dist[v],\ dist[u] + w(u, v))$$

**复杂度**：$O((V + E) \log V)$（堆优化）。

**为什么 Dijkstra 正确**：非负权下，堆顶的 `dist` 已不可能再被其他路径改进（任何绕路都 ≥ 当前值）——贪心成立。

> ⚠️ 有**负权**必须用 Bellman-Ford / SPFA；Dijkstra 会错。

In [ ]:
import heapq

def network_delay_time(times, n, k):
    # 返回从 k 到所有节点的最短时间最大值；不可达返回 -1
    adj = [[] for _ in range(n + 1)]
    for u, v, w in times:
        adj[u].append((v, w))
    dist = [float('inf')] * (n + 1)
    dist[k] = 0
    heap = [(0, k)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue              # 过期记录，跳过
        for v, w in adj[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    ans = max(dist[1:])
    return -1 if ans == float('inf') else ans

print('test1:', network_delay_time([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2))   # 2
print('test2:', network_delay_time([[1, 2, 1]], 2, 1))                         # 1
print('test3:', network_delay_time([[1, 2, 1]], 2, 2))                         # -1（节点1不可达）

## §8 前缀树 Trie（LeetCode 208）

**题目**：实现 insert / search / startsWith。

**结构**：树节点存「子节点字典 + 是否结尾」：

```python
class TrieNode:
    def __init__(self):
        self.children = {}      # char -> TrieNode
        self.is_end = False
```

**复杂度**：$O(L)$（L 为单词长度）时间/空间每次操作。

**优势**：前缀查询 $O(L)$ 而非遍历全部单词；自动机/词频统计/AC 自动机的基础。

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True

    def _find(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

    def search(self, word):
        node = self._find(word)
        return node is not None and node.is_end

    def starts_with(self, prefix):
        return self._find(prefix) is not None

trie = Trie()
trie.insert('apple')
print('search(apple):', trie.search('apple'))      # True
print('search(app):', trie.search('app'))          # False（未标记结尾）
print('starts_with(app):', trie.starts_with('app'))  # True
trie.insert('app')
print('search(app) 后:', trie.search('app'))       # True
print('search(bad):', trie.search('bad'))          # False

## §9 复杂度总表（背熟）

| 模板 | 时间 | 空间 | 适用 |
|------|------|------|------|
| BFS/DFS | $O(V+E)$ | $O(V)$ | 遍历、连通块、无权最短路 |
| 网格 DFS/BFS | $O(RC)$ | $O(RC)$ | 岛屿类、感染类 |
| Kahn 拓扑 | $O(V+E)$ | $O(V)$ | 判环、排序 |
| 并查集 | $O(\alpha(n))$ | $O(n)$ | 动态连通 |
| Dijkstra（堆） | $O((V+E)\log V)$ | $O(V+E)$ | 非负权最短路 |
| Trie | $O(L)$/操作 | $O(\sum L)$ | 前缀查询 |

### 选型速查（面试 30 秒）

- 无权最短路 → BFS；有权非负 → Dijkstra；有负权 → Bellman-Ford
- 判环：有向 → 拓扑/Kahn；无向 → 并查集/DFS
- 连通块 → DFS/BFS/并查集；单词前缀 → Trie

## §10 常见 bug 与边界（面试踩坑区）

- **visited 缺失**：无向图 DFS 不标记 → 死循环（或指数爆炸）
- **BFS 入队即标记**：先标记再入队；否则同一节点可能入队多次
- **Dijkstra 忘了跳过过期记录**：堆里有旧距离记录，`d > dist[u]` 必须 continue
- **拓扑：入度数组的边方向**：`prerequisites` 是 `[a, b]` 表示 b→a 还是 a→b，先确认
- **网格越界**：四方向前先 `0 <= ni < R and 0 <= nj < C`
- **并查集路径压缩写错**：`parent[x] = parent[parent[x]]` 或递归版都行，但别写漏
- **Trie 的 is_end**：search 判结尾、starts_with 不判——混了必错
- **图不连通**：Dijkstra 结果里可能有 `inf`，记得处理（返回 -1）

## §11 面试追问与扩展（加分项）

### 11.1 BFS 为什么能找到最短路径（无权图）

BFS 按层扩展：第 $k$ 层 = 距离 $k$。若存在更短路径，其终点会在更早的层被访问——首次访问即最短，无需反复比较。

### 11.2 Dijkstra 与 BFS 的关系

Dijkstra = BFS 的推广：队列换成优先队列、按累计距离排序。同理可推：无权图 Dijkstra 退化为 BFS。

### 11.3 进阶话题（了解即可）

- Floyd-Warshall：全源最短路 $O(V^3)$（n ≤ 几百时用）
- 二分图判定：BFS 染色（785）
- 欧拉路径：一笔画（332）
- 强连通分量：Tarjan（竞赛级）

## §12 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（10 题）

- [ ] 1. 岛屿数量（LeetCode 200）
- [ ] 2. 腐烂的橘子（994）
- [ ] 3. 课程表（207）
- [ ] 4. 课程表 II（210，输出拓扑序）
- [ ] 5. 实现 Trie（前缀树）（208）
- [ ] 6. 省份数量（547）
- [ ] 7. 二叉树的层序遍历（102，BFS 热身）
- [ ] 8. 图像渲染（733，连通块）
- [ ] 9. 被围绕的区域（130）
- [ ] 10. 钥匙和房间（841，DFS 可达性）

### L2 进阶（10 题）

- [ ] 11. 网络延迟时间（743，Dijkstra）
- [ ] 12. 冗余连接（684，并查集）
- [ ] 13. 单词搜索（79，网格 DFS）
- [ ] 14. 单词接龙（127，BFS 最短转换）
- [ ] 15. 判断二分图（785）
- [ ] 16. 找到小镇的法官（997，出度入度）
- [ ] 17. 太平洋大西洋水流问题（417，反向 DFS）
- [ ] 18. 矩阵中的最长递增路径（329，DFS+记忆化）
- [ ] 19. 重新安排行程（332，欧拉路径）
- [ ] 20. 连坐/可能的二分法（886，染色）

### L3 挑战（5 题）

- [ ] 21. 最小生成树（1584，Prim/Kruskal 模板）
- [ ] 22. 网络延迟时间的全源版（Floyd）
- [ ] 23. 马踏棋盘/骑士概率（688，DP+BFS）
- [ ] 24. 课程表 III（630，贪心+堆）
- [ ] 25. 最小高度树（310，拓扑剥皮）

### 自测要点

- 5 分钟内盲写：BFS 模板 + Kahn 拓扑 + 并查集模板
- 讲清 BFS 最短路径原理与 Dijkstra 贪心正确性
- 网格题统一用「sink 原地标记」或 visited 数组，能说清二者取舍

## 附录：参考与下节预告

- 《算法导论》第 22 章（BFS/DFS）、23 章（最小生成树）、24 章（最短路）
- LeetCode 分类题单：图 / 并查集 / Trie

### 下节预告：主题 11-13 — 堆与 Top-K / 位运算 / 字符串（收尾三连）

> 💡 本主题验收：`高频题单.md`「图」一节全部打勾，能手写并查集模板（含路径压缩）。